In [27]:
# imports
import pandas as pd
import numpy as np
import sweetviz as sv

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import balanced_accuracy_score, accuracy_score, f1_score, classification_report, confusion_matrix


from sklearn.ensemble import GradientBoostingClassifier, AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

from xgboost import XGBClassifier


In [2]:
# importing data
# Update these file paths if your folder location is different
train = pd.read_csv(r"C:\Users\2003n\OneDrive - Cal Poly\Desktop\playground-series-s6e4\train.csv")
test = pd.read_csv(r"C:\Users\2003n\OneDrive - Cal Poly\Desktop\playground-series-s6e4\test.csv")
sample = pd.read_csv(r"C:\Users\2003n\OneDrive - Cal Poly\Desktop\playground-series-s6e4\sample_submission.csv")

train.head()


,id,Soil_Type,Soil_pH,Soil_Moisture,Organic_Carbon,Electrical_Conductivity,Temperature_C,Humidity,Rainfall_mm,Sunlight_Hours,...,Crop_Type,Crop_Growth_Stage,Season,Irrigation_Type,Water_Source,Field_Area_hectare,Mulching_Used,Previous_Irrigation_mm,Region,Irrigation_Need
0,0,Loamy,4.92,32.58,1.01,3.05,15.01,50.61,725.99,5.90,...,Sugarcane,Sowing,Zaid,Drip,Rainwater,0.82,No,112.16,East,Low
1,1,Clay,7.08,56.61,0.44,2.00,22.92,67.86,985.66,6.98,...,Wheat,Vegetative,Kharif,Rainfed,River,5.27,Yes,47.16,South,Low
2,2,Clay,5.69,27.71,0.81,2.83,26.97,92.22,2201.70,6.05,...,Rice,Vegetative,Kharif,Sprinkler,Reservoir,8.24,Yes,110.38,North,Low
3,3,Sandy,5.65,13.32,1.33,0.87,13.32,61.57,1357.33,9.12,...,Wheat,Flowering,Kharif,Canal,River,8.32,Yes,53.85,South,Medium
4,4,Clay,7.96,59.14,0.38,0.96,20.22,91.11,1538.20,6.95,...,Wheat,Sowing,Rabi,Canal,River,7.37,No,93.19,South,Low


In [4]:
# quick EDA checks
print("Train shape:", train.shape)
print("Test shape:", test.shape)
print("Missing values:")
print(train.isna().sum())
print("Duplicates:", train.duplicated().sum())
print("Target counts:")
print(train["Irrigation_Need"].value_counts())
print("Target percentages:")
print(train["Irrigation_Need"].value_counts(normalize=True))


Train shape: (630000, 21)
Test shape: (270000, 20)
Missing values:
id                         0
Soil_Type                  0
Soil_pH                    0
Soil_Moisture              0
Organic_Carbon             0
Electrical_Conductivity    0
Temperature_C              0
Humidity                   0
Rainfall_mm                0
Sunlight_Hours             0
Wind_Speed_kmh             0
Crop_Type                  0
Crop_Growth_Stage          0
Season                     0
Irrigation_Type            0
Water_Source               0
Field_Area_hectare         0
Mulching_Used              0
Previous_Irrigation_mm     0
Region                     0
Irrigation_Need            0
dtype: int64
Duplicates: 0
Target counts:
Irrigation_Need
Low       369917
Medium    239074
High       21009
Name: count, dtype: int64
Target percentages:
Irrigation_Need
Low       0.587170
Medium    0.379483
High      0.033348
Name: proportion, dtype: float64


In [5]:
# optional Sweetviz EDA report
report = sv.analyze(train)
report.show_html("hw3_irrigation_eda_report.html")


                                             |          | [  0%]   00:00 -> (? left)

Report hw3_irrigation_eda_report.html was generated! NOTEBOOK/COLAB USERS: the web browser MAY not pop up, regardless, the report IS saved in your notebook/colab files.


In [7]:
# preprocessing shared by sklearn boosting models
# GradientBoostingClassifier and AdaBoostClassifier need numeric features, so I use get_dummies.

le = LabelEncoder()
y = le.fit_transform(train["Irrigation_Need"])

X = train.drop(["Irrigation_Need"], axis=1)
X_test_final = test.copy()

full = pd.concat([X, X_test_final], axis=0)
full_encoded = pd.get_dummies(full, drop_first=False)

X_encoded = full_encoded.iloc[:len(X), :]
X_test_encoded = full_encoded.iloc[len(X):, :]

X_train, X_valid, y_train, y_valid = train_test_split(
    X_encoded, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

print(X_train.shape, X_valid.shape, X_test_encoded.shape)


(504000, 44) (126000, 44) (270000, 44)


In [13]:
# Model 1: Gradient Boosting Classifier with 3 hyperparameter sets

gb_param_sets = {
    "GB Set 1": {
        "n_estimators": 50,
        "learning_rate": 0.10,
        "max_depth": 3,
        "subsample": 1.0
    },
    "GB Set 2": {
        "n_estimators": 100,
        "learning_rate": 0.05,
        "max_depth": 3,
        "subsample": 0.8
    },
    "GB Set 3": {
        "n_estimators": 75,
        "learning_rate": 0.08,
        "max_depth": 5,
        "subsample": 0.8
    }
}

gb_results = []
gb_models = {}

for name, params in gb_param_sets.items():
    print(f"Training {name}...")

    model = GradientBoostingClassifier(
        random_state=42,
        **params
    )

    model.fit(X_train, y_train)
    valid_pred = model.predict(X_valid)

    gb_models[name] = model

    gb_results.append({
        "model": name,
        "valid_bal_acc": balanced_accuracy_score(y_valid, valid_pred),
        "valid_accuracy": accuracy_score(y_valid, valid_pred),
        "valid_macro_f1": f1_score(y_valid, valid_pred, average="macro")
    })

gb_results_df = pd.DataFrame(gb_results).sort_values("valid_bal_acc", ascending=False)
gb_results_df


Training GB Set 1...
Training GB Set 2...
Training GB Set 3...


,model,valid_bal_acc,valid_accuracy,valid_macro_f1
2,GB Set 3,0.964043,0.986071,0.972002
0,GB Set 1,0.951203,0.983429,0.964615
1,GB Set 2,0.920681,0.980429,0.945311


In [10]:
# best Gradient Boosting model validation report
best_gb_name = gb_results_df.iloc[0]["model"]
best_gb = gb_models[best_gb_name]
y_pred_gb = best_gb.predict(X_valid)

print("Best Gradient Boosting model:", best_gb_name)
print("Validation balanced accuracy:", balanced_accuracy_score(y_valid, y_pred_gb))
print("Validation accuracy:", accuracy_score(y_valid, y_pred_gb))
print("Validation macro F1:", f1_score(y_valid, y_pred_gb, average="macro"))
print(classification_report(y_valid, y_pred_gb, target_names=le.classes_))


Best Gradient Boosting model: GB Set 3 - deeper trees
Validation balanced accuracy: 0.9640429724442824
Validation accuracy: 0.9860714285714286
Validation macro F1: 0.9720024935997098
              precision    recall  f1-score   support

        High       0.97      0.92      0.94      4202
         Low       0.99      1.00      0.99     73983
      Medium       0.99      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.99      0.99      0.99    126000



In [14]:
# Gradient Boosting Kaggle submission
gb_test_pred = best_gb.predict(X_test_encoded)
gb_test_labels = le.inverse_transform(gb_test_pred)

gb_submission = sample.copy()
gb_submission["Irrigation_Need"] = gb_test_labels
gb_submission.to_csv("hw3_gradient_boosting_submission.csv", index=False)
gb_submission.head()


,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


In [28]:
# preprocessing for LightGBM
# LightGBM can handle categorical columns when they are set as category dtype.

lgbm_train = train.copy()
lgbm_test = test.copy()

le_lgbm = LabelEncoder()
y_lgbm = le_lgbm.fit_transform(lgbm_train["Irrigation_Need"])
X_lgbm = lgbm_train.drop(["Irrigation_Need"], axis=1)

for col in X_lgbm.select_dtypes(include="object").columns:
    X_lgbm[col] = X_lgbm[col].astype("category")
    lgbm_test[col] = lgbm_test[col].astype("category")

X_train_lgbm, X_valid_lgbm, y_train_lgbm, y_valid_lgbm = train_test_split(
    X_lgbm,
    y_lgbm,
    test_size=0.2,
    random_state=42,
    stratify=y_lgbm
)

print(X_train_lgbm.shape, X_valid_lgbm.shape, lgbm_test.shape)


(504000, 20) (126000, 20) (270000, 20)


In [34]:
import sys
!{sys.executable} -m pip install lightgbm

  Using cached lightgbm-4.6.0-py3-none-win_amd64.whl.metadata (17 kB)
Using cached lightgbm-4.6.0-py3-none-win_amd64.whl (1.5 MB)


In [35]:
from lightgbm import LGBMClassifier

In [36]:
# Model 2: LightGBM Classifier with 3 hyperparameter sets

lgbm_param_sets = {
    "LGBM Set 1": {
        "n_estimators": 100,
        "learning_rate": 0.10,
        "max_depth": 3,
        "subsample": 1.0,
        "colsample_bytree": 1.0
    },
    "LGBM Set 2": {
        "n_estimators": 200,
        "learning_rate": 0.05,
        "max_depth": 3,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    },
    "LGBM Set 3": {
        "n_estimators": 150,
        "learning_rate": 0.08,
        "max_depth": 5,
        "subsample": 0.8,
        "colsample_bytree": 0.8
    }
}

lgbm_results = []
lgbm_models = {}

for name, params in lgbm_param_sets.items():
    print(f"Training {name}...")

    model = LGBMClassifier(
        random_state=42,
        verbose=-1,
        **params
    )

    model.fit(X_train_lgbm, y_train_lgbm)
    valid_pred = model.predict(X_valid_lgbm)

    lgbm_models[name] = model

    lgbm_results.append({
        "model": name,
        "valid_bal_acc": balanced_accuracy_score(y_valid_lgbm, valid_pred),
        "valid_accuracy": accuracy_score(y_valid_lgbm, valid_pred),
        "valid_macro_f1": f1_score(y_valid_lgbm, valid_pred, average="macro")
    })

lgbm_results_df = pd.DataFrame(lgbm_results).sort_values("valid_bal_acc", ascending=False)
lgbm_results_df

Training LGBM Set 1...
Training LGBM Set 2...
Training LGBM Set 3...


,model,valid_bal_acc,valid_accuracy,valid_macro_f1
2,LGBM Set 3,0.964099,0.985135,0.970722
1,LGBM Set 2,0.958341,0.984127,0.967893
0,LGBM Set 1,0.958315,0.984246,0.968259


In [37]:
# best LightGBM model validation report

best_lgbm_name = lgbm_results_df.iloc[0]["model"]
best_lgbm = lgbm_models[best_lgbm_name]

# use the same validation data structure that the model was trained with
y_pred_lgbm = best_lgbm.predict(X_valid_lgbm)

print("Best LightGBM model:", best_lgbm_name)
print("Validation balanced accuracy:", balanced_accuracy_score(y_valid_lgbm, y_pred_lgbm))
print("Validation accuracy:", accuracy_score(y_valid_lgbm, y_pred_lgbm))
print("Validation macro F1:", f1_score(y_valid_lgbm, y_pred_lgbm, average="macro"))
print(classification_report(y_valid_lgbm, y_pred_lgbm))

Best LightGBM model: LGBM Set 3
Validation balanced accuracy: 0.9640992611484854
Validation accuracy: 0.9851349206349206
Validation macro F1: 0.9707221828805835
              precision    recall  f1-score   support

           0       0.96      0.92      0.94      4202
           1       0.99      0.99      0.99     73983
           2       0.98      0.98      0.98     47815

    accuracy                           0.99    126000
   macro avg       0.98      0.96      0.97    126000
weighted avg       0.99      0.99      0.99    126000



In [38]:
# LightGBM Kaggle submission

lgbm_test_pred = best_lgbm.predict(lgbm_test)
lgbm_test_labels = le_lgbm.inverse_transform(lgbm_test_pred)

lgbm_submission = sample.copy()
lgbm_submission["Irrigation_Need"] = lgbm_test_labels
lgbm_submission.to_csv("hw3_lightgbm_submission.csv", index=False)

lgbm_submission.head()

,id,Irrigation_Need
0,630000,Low
1,630001,Low
2,630002,Low
3,630003,Low
4,630004,Low


I tried Gradient Boosting and LightGBM for my additional models. I started with a baseline/generic parameters for each of the models. I then used 2 other parameters that were either slower learnigns rates with more estimators and another with more deeper trees. For both models, the best setup was the one with deeper trees with GB geeting a balanced accuracy of 0.9640 and LGBM getting a balanced accuracy of 0.9641. For gradient boosting, my 2nd model set up was weaker than the rest with a balanced accuracy of 0.9207. This showed me that increasing the number of trees was not always the best way to increase performanced of my modeks. 

Both models in their best setups had extremely similar results, with LGBM having a slight edge in balanced accuracy, but Gradient Boosting having a greater edge in overall accuracy and F1 score. LGBM was definitely faster to run in comparison to Gradient Boosting though. On Kaggle however, Gradient Boosting had a better public score of 0.95879 vs LGBM of 0.95796.

Phase 1 Leaderboard Scores:
Random Forest = 0.96211
XGB = 0.95925

Phase 2
Gradient Boosting = 0.95879
Light GBM = 0.95796